In [15]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings



In [ ]:
load_dotenv()
os.chdir("../")
data_path = "data"

In [2]:
def load_docs(data_path):
    loader = DirectoryLoader(
        data_path, glob="*.pdf", loader_cls=PyPDFLoader  # type: ignore
    )
    documents = loader.load()
    return documents


documents = load_docs(data_path)

In [3]:
documents[:5]

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 0, 'page_label': 'i'}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 1, 'page_label': 'ii'}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular.

In [4]:
documents[5]

Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 5, 'page_label': 'vi'}, page_content='Chapter 4\nHOW TO TAKE CARE OF A SICK PERSON . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39\nThe Comfort of the Sick Person 39 Watching for Changes 41\nSpecial Care for a Person Who Is Very Ill 40 Signs of Dangerous Illness 42\nLiquids 40 When and How to Look for Medical Help 43\nFood 41 What to Tell the Health Worker 43\nCleanliness and Changing Position in Bed 41 Patient Report 44\nChapter 5\nHEALING WITHOUT MEDICINES . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 45\nHealing with Water 46\nWhen Water Is Better than Medicines 47\nChapter 6\nRIGHT AND WRONG USE OF MODERN MEDICINES. . . . . . . . . . . . . . . . . . . . . 

In [5]:
documents[5].metadata.keys()

dict_keys(['producer', 'creator', 'creationdate', 'moddate', 'trapped', 'source', 'total_pages', 'page', 'page_label'])

In [6]:
vars(documents[5])

{'id': None,
 'metadata': {'producer': 'Adobe PDF Library 9.0',
  'creator': 'Acrobat 9.3.3',
  'creationdate': '2010-07-24T16:57:37-07:00',
  'moddate': '2010-07-28T15:49:40-07:00',
  'trapped': '/False',
  'source': 'data/WhereThereIsNoDoctor.pdf',
  'total_pages': 503,
  'page': 5,
  'page_label': 'vi'},
 'page_content': 'Chapter 4\nHOW TO TAKE CARE OF A SICK PERSON . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39\nThe Comfort of the Sick Person 39 Watching for Changes 41\nSpecial Care for a Person Who Is Very Ill 40 Signs of Dangerous Illness 42\nLiquids 40 When and How to Look for Medical Help 43\nFood 41 What to Tell the Health Worker 43\nCleanliness and Changing Position in Bed 41 Patient Report 44\nChapter 5\nHEALING WITHOUT MEDICINES . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 45\nHealing with Water 46\nWhen Water Is Better than Medicines 47\nChapter 6\nRIGHT AND WRONG USE OF MODERN MEDICINES. . . . . . . 

In [7]:
def filter_extracted_docs_content(docs):
    return [
        Document(
            page_content=doc.page_content,
            metadata={
                "source": doc.metadata.get("source"),
                "page": doc.metadata.get("page"),
            },
        )
        for doc in docs
    ]

In [8]:
filtered_docs = filter_extracted_docs_content(documents)

In [9]:
filtered_docs[:5]

[Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 0}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.\n2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED WITH  YOUR HELP.  \nIf you are a community health worker, doctor, mother, or anyone with ideas or \nsuggest

In [10]:
len(filtered_docs)

503

In [11]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=25)
    chunked_documents = text_splitter.split_documents(documents)
    return chunked_documents


chunked_documents = create_chunks(filtered_docs)

In [12]:
chunked_documents[:5]

[Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 0}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED WITH  YOUR HELP.

In [13]:
len(chunked_documents)

2519

In [16]:
def setup_hf_cache():
    base_dir = os.getcwd()
    hf_home = os.path.join(base_dir, "embedding_model")
    os.makedirs(hf_home, exist_ok=True)

    os.environ["HF_HOME"] = hf_home
    os.environ["TRANSFORMERS_CACHE"] = hf_home
    os.environ["HF_DATASETS_CACHE"] = hf_home

    return hf_home


##############
def get_embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


hf_cache_path = setup_hf_cache()
embedding_model = get_embedding_model()

In [18]:
from langchain_community.vectorstores import FAISS

# store embedding in chroma/fiass

DB_FIASS_PATH = "vectorstore/db_fiass"
db = FAISS.from_documents(chunked_documents, embedding_model)
db.save_local(DB_FIASS_PATH)